In [1]:
import os
import urllib
import json
import urllib

def download_and_load(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(text_data)        

    with open(file_path, "r") as file:
        data = json.load(file)
    return data

In [2]:
file_path =  "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

In [3]:
data = download_and_load(file_path, url)

In [4]:
print(len(data))

1100


In [5]:
print(data[50])

{'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [6]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else "")
    return instruction_text + input_text

In [7]:
model_input = format_input(data[50])
desired_output = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_output)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [8]:
model_input = format_input(data[999])
desired_output = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_output)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


In [9]:
# dividing the data into training(85%), validation(5%), testing(10%)
train_portion = int(len(data) * .85)
test_portion = int(len(data) * .1)
val_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("training set length:", len(train_data))
print("validation set length:", len(val_data))
print("testing set length:", len(test_data))

training set length: 935
validation set length: 55
testing set length: 110


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.encoded_texts = []
        for text in data:
            instruction_plus_input = format_input(text)
            response = f"\n\n### Response:\n{text['output']}"
            full_text = instruction_plus_input + response
            self.encoded_texts.append(tokenizer.encode(full_text))
        
    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


In [11]:
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):

    batch_max_length = max(len(i)+1 for i in batch)
    input_lst, target_lst = [], []
    for item in batch:
        token_lst = item.copy()
        token_lst += [pad_token_id]
        padded = token_lst + [pad_token_id] * (batch_max_length - len(token_lst))
        input_tokens = torch.tensor(padded[:-1])
        target_tokens = torch.tensor(padded[1:])
        mask = target_tokens == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            target_tokens[indices[1:]] = ignore_index
        if allowed_max_length is not None:
            input_tokens = input_tokens[:allowed_max_length]
            target_tokens = target_tokens[:allowed_max_length]
        input_lst.append(input_tokens)
        target_lst.append(target_tokens)

    inputs_tensor = torch.stack(input_lst).to(device)
    target_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor, target_tensor

In [12]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

In [13]:
input, target = custom_collate_fn(batch)
print(input)
print(target)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [15]:
from functools import partial

customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)

In [16]:
from numpy import test
from torch.utils.data import DataLoader
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

validation_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [17]:
for input, target in train_loader:
    print(input.shape,target.shape)

torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.Size([8, 68])


In [18]:
# lets load the pretrained medium sized model
import sys
from pathlib import Path
project_root = Path.cwd().parent.parent.parent
sys.path.append(str(project_root))

from pretraining.src.gpt_download import download_and_load_gpt2
BASE_CONFIG = {
    "vocab_size": 50257,         
    "context_length": 1024,      
    "drop_rate": 0.0,            
    "qkv_bias": True             
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size, 
    models_dir="gpt2"
)

File already exists and is up-to-date: gpt2\355M\checkpoint
File already exists and is up-to-date: gpt2\355M\encoder.json
File already exists and is up-to-date: gpt2\355M\hparams.json
File already exists and is up-to-date: gpt2\355M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\355M\model.ckpt.index
File already exists and is up-to-date: gpt2\355M\model.ckpt.meta
File already exists and is up-to-date: gpt2\355M\vocab.bpe


In [19]:
from gpt_for_text_generation.src.gpt import GPTModel
from pretraining.src.gpt_generate import load_weights_into_gpt

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [20]:
# test the knowledge of pretrained model
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [21]:
from pretraining.src.gpt_generate import generate, token_ids_to_text, text_to_token_ids

token_ids = generate(model, idx=text_to_token_ids(input_text, tokenizer), max_new_tokens=35, context_size=BASE_CONFIG["context_length"])
generated_text = token_ids_to_text(token_ids, tokenizer)
print(generated_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Response:

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the


In [22]:
response_text = generated_text[len(input_text):].strip()
print(response_text)

### Response:

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the


In [23]:
#finetuning the model 
from pretraining.src.pretrain import calc_loss_loader, train_model_simple
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
print("training loss", train_loss)
print("validation loss", val_loss)

training loss tensor(3.8249)
validation loss tensor(3.7611)


In [24]:
# i will not train it here, instead i will train it on kaggle and from there i will download the .pth file
# import tiktoken
# tokenizer = tiktoken.get_encoding("gpt2")
# torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
# num_epochs = 2

# train_loss, val_loss, tokens_seen = train_model_simple(
#     model, train_loader, val_loader, optimizer, device, num_epochs=num_epochs, eval_freq=5, eval_iter=5, start_context=format_input(val_data[0]), tokenizer=tokenizer
# )

In [25]:
file = torch.load("model_and_optimizer_instruction.pth", map_location=device)
model.load_state_dict(file["model_state_dict"])
optimizer.load_state_dict(file["optimizer_state_dict"])
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [26]:
# lets test the finetuned model on the first 3 test set text using generate function
for text in test_data[:3]:
    input_text = format_input(text)
    token_ids = generate(model, idx=text_to_token_ids(input_text, tokenizer).to(device), max_new_tokens=256, context_size=BASE_CONFIG["context_length"],eos_id=50256)
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].strip().replace("### Response:", "")
    print(input_text)
    print(f"\nCorrect response:\n>> {text['output']}")
    print(f"\Model response:\n>> {response_text.strip()}")
    

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.

Correct response:
>> The car is as fast as lightning.
\Model response:
>> The car is as fast as a cheetah.
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What type of cloud is typically associated with thunderstorms?

Correct response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.
\Model response:
>> The type of cloud associated with thunderstorms is a cumulus cloud.
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Name the author of 'Pride and Prejudice'.

Correct response:
>> Jane Austen.
\Model response:
>> The author of 'Pride and Prejudice' is Jane Austen.


In [ ]:
# from tqdm import tqdm

# for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
#     input_text = format_input(entry)
#     token_ids = generate(model, idx=text_to_token_ids(input_text, tokenizer), max_new_tokens=256, context_size=BASE_CONFIG["context_length"], eos_id=50256)
#     generated_text = token_ids_to_text(token_ids, tokenizer)
#     model_response = generated_text[len(input_text):].replace("### Response:","").strip()
#     test_data[i]['model_response'] = model_response

# with open("instruction-data-with-response.json", "w") as file:
#     json.dump(test_data, file, indent=4)


  6%|▋         | 7/110 [00:48<11:54,  6.94s/it]


KeyboardInterrupt: 

In [28]:
import psutil
def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running
ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError(
"Ollama not running. Launch ollama before proceeding."
)
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [29]:
with open("instruction-data-with-response.json", "r") as file:
    test_data = json.load(file)

In [30]:
import requests

def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {
            "seed": 123, 
            "temperature": 0,
            "num_ctx": 2048
        },
    }

    with requests.post(url=url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines():
            if not line:
                continue
            response_json = json.loads(line)
            if 'message' in response_json:
                response_data += response_json['message']['content']

    return response_data

In [31]:
result = query_model("hello llama")
print(result)

Hello there! It's not every day I get to chat with a llama! How are you doing today?


In [32]:
# now lets give score to each response of our model by prompting it to the llama using api call(query_model)
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )

    print("\nDataset response:")
    print(">>", entry["output"])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n----------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a cheetah.

Score:
>> I'd rate the model response "The car is as fast as a cheetah." an 85 out of 100.

Here's why:

* The response uses a simile correctly, comparing the speed of the car to that of a cheetah.
* The comparison is relevant and makes sense, as both cars and cheetahs are known for their speed.
* The phrase "as fast as" is used consistently with the original instruction.

The only reason I wouldn't give it a perfect score is that lightning is often used as an example of extremely rapid movement in English language, so using a more common or relatable comparison like a cheetah is still a good choice. However, if the goal was to exactly replicate the original response's level of speed and vividness, I might deduct a few points for not using lightning specifically.

----------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumu

In [33]:
from tqdm import tqdm
def generate_model_scores(json_data, model="llama3"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry['model_response']}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."  
        )

        score = query_model(prompt)
        try:
            scores.append(int(score))
        except ValueError:
            continue
    return scores